# Simple language first, with real data

Run 2 of the goals experiment showed that learning a skill in a small, clean language first
made a small model learn it in English from far fewer examples. This notebook tries the same
recipe with real text instead of a toy world:

| stage | `simple-first` | `english` (fair rival) | `ceiling` |
|---|---|---|---|
| 1 | real toki pona (Tatoeba, Wikipedia, lipu kule) | the English budget | fresh English, unlimited |
| 2 | bridge: toki pona ⇄ English sentence pairs, plus English stories with Basic English words swapped to their toki pona hub word | (same English budget) | (fresh English) |
| 3 | the English budget (+ a little toki pona replay) | (same English budget) | (fresh English) |

The English budget is a small, fixed set of SimpleStories plus the English half of the Tatoeba pairs.
`simple-first` and `english` see exactly the same English and train for the same number of steps;
the only difference is that `simple-first` spends part of its steps on toki pona instead of
repeating English. `ceiling` gets unlimited fresh English as an upper bound. For every model the
checkpoint with the best English validation loss is kept, so repeating data can't unfairly hurt `english`.

**Tests**: English validation loss, BLiMP (67 grammar tests, 1,000 pairs each), and bAbI
(20 reasoning tasks; every model gets the same short fine-tune on bAbI, then is tested).

**On Kaggle**: Settings → Accelerator → GPU T4; Settings → **Internet on** (needed to download the
datasets; it needs a phone-verified account). Then *Save Version → Save & Run All* so it runs in the
background (about 2–2.5 hours), and download `real_export.zip` and `REAL_RESULTS.md` from the output.

In [ ]:
import json, math, os, random, re, statistics as st, struct, sys, time, zipfile
import torch
import torch.nn as nn
import torch.nn.functional as F

try:
    import sentencepiece as spm
except ImportError:
    os.system("pip install -q sentencepiece")
    import sentencepiece as spm
from datasets import load_dataset

QUICK = "--quick" in sys.argv
DEV = "cuda" if torch.cuda.is_available() else "cpu"
AMP = dict(device_type=DEV, dtype=torch.float16 if DEV == "cuda" else torch.bfloat16, enabled=DEV == "cuda")

if QUICK:
    D, LAYERS, HEADS, FFN, CTX, VOCAB = 64, 2, 2, 128, 128, 2000
    S1, S2, S3, BATCH, EVAL_EVERY = 60, 40, 60, 8, 40
    N_STORIES, N_CEIL, BABI_STEPS, BLIMP_N, BABI_N = 300, 3000, 30, 20, 10
else:
    D, LAYERS, HEADS, FFN, CTX, VOCAB = 384, 8, 6, 1024, 256, 8192   # about 17M parameters
    S1, S2, S3, BATCH, EVAL_EVERY = 1500, 1500, 3000, 64, 500
    N_STORIES, N_CEIL, BABI_STEPS, BLIMP_N, BABI_N = 10_000, 400_000, 1500, 1000, 200
LR = 1e-3
SEED = 0
random.seed(SEED); torch.manual_seed(SEED)
T0 = time.time()


def log(*a):
    print(f"[{(time.time() - T0) / 60:5.1f} min]", *a, flush=True)

## Data

In [ ]:
def clean(s):
    return " ".join((s or "").split())


log("loading toki pona")
tato = load_dataset("NetherQuartz/tatoeba-tokipona", split="train")
pairs = [(clean(r["tok"]), clean(r["en"])) for r in tato if r["tok"] and r["en"]]
mined = load_dataset("NetherQuartz/tokipona-mined-pairs", split="train")
mono = load_dataset("NetherQuartz/tokipona-monolingual", split="train")
tp_text = list(dict.fromkeys([p[0] for p in pairs] + [clean(r["tok"]) for r in mined] + [clean(r["text"]) for r in mono]))
random.shuffle(pairs); random.shuffle(tp_text)
if QUICK:
    pairs, tp_text = pairs[:2000], tp_text[:4000]
tp_val, tp_text = tp_text[:500], tp_text[500:]

log("loading SimpleStories")
ss = load_dataset("SimpleStories/SimpleStories", split="train", streaming=True)
stories = []
for r in ss:
    stories.append(clean(r["story"]))
    if len(stories) >= N_STORIES + N_CEIL + 2000:
        break
val_stories, budget, ceil_pool = stories[:2000], stories[2000:2000 + N_STORIES], stories[2000 + N_STORIES:]
en_budget = budget + [p[1] for p in pairs]          # the English both rivals get

HUBS = json.loads('{"table":{"i":"mi","across":"tawa","after":"tenpo","again":"sin","against":"utala","all":"ale","almost":"lili","among":"insa","any":"ale","before":"tenpo","between":"insa","come":"kama","down":"anpa","east":"nasin","enough":"mute","even":"sama","ever":"tenpo","every":"ale","far":"weka","forward":"tawa","get":"kama","give":"pana","go":"tawa","he":"ona","here":"lon","keep":"awen","little":"lili","make":"pali","may":"ken","much":"mute","near":"lon","north":"nasin","now":"tenpo","off":"pini","only":"taso","other":"ante","out":"weka","over":"pini","put":"pana","say":"toki","see":"lukin","seem":"sama","send":"pana","some":"mute","south":"nasin","still":"awen","such":"sama","take":"kama","there":"ni","though":"taso","through":"insa","till":"tawa","together":"wan","tomorrow":"tenpo","under":"anpa","very":"mute","well":"pona","west":"nasin","when":"tenpo","where":"lon","while":"la","who":"jan","why":"tan","will":"wile","yes":"lon","yesterday":"tenpo","you":"sina","account":"toki","act":"pali","addition":"namako","adjustment":"ante","advertisement":"sitelen","agreement":"pilin","air":"kon","amount":"mute","amusement":"musi","animal":"soweli","answer":"toki","apparatus":"ilo","approval":"pona","argument":"toki","art":"musi","attack":"utala","attention":"lukin","attraction":"olin","authority":"lawa","back":"monsi","balance":"sama","base":"noka","behavior":"nasin","belief":"pilin","birth":"kama","bit":"lili","bite":"uta","blood":"telo","blow":"utala","body":"sijelo","brass":"kiwen","bread":"pan","breath":"kon","brother":"jan","building":"tomo","burn":"seli","burst":"pakala","business":"esun","butter":"ko","canvas":"len","cause":"tan","chalk":"ilo","chance":"ken","change":"ante","cloth":"len","coal":"kiwen","comfort":"pona","committee":"kulupu","company":"kulupu","comparison":"sama","competition":"utala","condition":"lon","connection":"linja","control":"lawa","cook":"seli","copper":"kiwen","copy":"sama","cork":"kiwen","cotton":"len","cough":"kalama","country":"ma","cover":"len","crack":"pakala","credit":"mani","crime":"pali","crush":"pakala","cry":"telo","current":"tenpo","curve":"sike","damage":"pakala","danger":"ike","daughter":"meli","day":"tenpo","death":"moli","debt":"mani","decision":"wile","degree":"nanpa","desire":"wile","destruction":"pakala","detail":"lili","development":"pali","digestion":"moku","direction":"nasin","discovery":"sona","discussion":"toki","disease":"jaki","disgust":"jaki","distance":"weka","distribution":"pana","division":"kipisi","doubt":"sona","drink":"moku","driving":"lawa","dust":"ko","earth":"ma","edge":"selo","education":"pana","effect":"tan","end":"pini","error":"pakala","event":"tenpo","example":"ijo","exchange":"esun","existence":"lon","expansion":"suli","experience":"pilin","expert":"jan","fact":"lon","fall":"anpa","family":"kulupu","father":"mama","fear":"monsuta","feeling":"pilin","fiction":"toki","field":"ma","fight":"utala","fire":"seli","flame":"seli","flight":"tawa","flower":"kasi","food":"moku","force":"wawa","form":"sijelo","friend":"pona","front":"sinpin","fruit":"kili","glass":"kiwen","government":"lawa","grain":"pan","grass":"kasi","grip":"jo","group":"kulupu","growth":"suli","guide":"lawa","harbor":"ma","harmony":"sama","hate":"olin","hearing":"kute","heat":"seli","help":"pana","history":"tenpo","hole":"lupa","hope":"wile","hour":"tenpo","humor":"musi","ice":"telo","idea":"pilin","impulse":"wile","increase":"kama","industry":"pali","insect":"pipi","instrument":"ilo","insurance":"mani","interest":"wile","invention":"pali","iron":"kiwen","jelly":"ko","journey":"nasin","judge":"jan","jump":"tawa","kick":"noka","kiss":"uta","knowledge":"sona","land":"ma","language":"toki","law":"lawa","lead":"lawa","learning":"kama","leather":"selo","letter":"lipu","level":"supa","lift":"sewi","light":"suno","limit":"selo","linen":"len","liquid":"telo","list":"lipu","loss":"weka","love":"olin","machine":"ilo","man":"mije","manager":"jan","mark":"sitelen","market":"esun","mass":"suli","meal":"moku","measure":"nanpa","meat":"moku","meeting":"kulupu","memory":"sona","metal":"kiwen","middle":"insa","milk":"telo","mind":"lawa","mine":"mi","minute":"tenpo","mist":"kon","money":"mani","month":"tenpo","morning":"tenpo","mother":"mama","motion":"tawa","mountain":"nena","move":"tawa","music":"kalama","name":"nimi","nation":"ma","need":"wile","news":"sona","night":"tenpo","noise":"kalama","note":"sitelen","number":"nanpa","observation":"lukin","offer":"pana","operation":"pali","opinion":"pilin","order":"nasin","organization":"kulupu","owner":"jan","page":"lipu","pain":"pilin","paint":"sitelen","paper":"lipu","part":"lili","paste":"ko","payment":"mani","peace":"pona","person":"jan","place":"ma","plant":"kasi","play":"musi","pleasure":"pilin","point":"lili","poison":"jaki","polish":"pona","porter":"jan","position":"lon","powder":"ko","power":"wawa","price":"mani","print":"sitelen","process":"nasin","produce":"pali","profit":"mani","property":"jo","prose":"toki","protest":"toki","pull":"tawa","punishment":"ike","purpose":"tan","push":"tawa","quality":"pona","question":"wile","rain":"telo","range":"suli","rate":"nanpa","ray":"linja","reaction":"pilin","reading":"lukin","reason":"tan","record":"sitelen","regret":"pilin","relation":"sama","religion":"nasin","representative":"jan","request":"wile","rest":"lape","reward":"pana","rhythm":"kalama","rice":"pan","river":"telo","road":"nasin","roll":"sike","room":"tomo","rub":"luka","rule":"lawa","run":"tawa","salt":"namako","sand":"ko","scale":"nanpa","science":"sona","sea":"telo","seat":"supa","secretary":"jan","selection":"anu","self":"mi","sense":"pilin","servant":"jan","sex":"unpa","shade":"pimeja","shake":"tawa","shame":"pilin","shock":"nasa","side":"poka","sign":"lipu","silk":"len","silver":"kiwen","sister":"jan","size":"suli","sky":"sewi","sleep":"lape","slip":"tawa","slope":"nena","smash":"pakala","smell":"kon","smile":"uta","smoke":"kon","sneeze":"kalama","snow":"ko","soap":"ko","society":"kulupu","son":"mije","song":"kalama","sort":"kule","sound":"kalama","soup":"moku","stage":"supa","start":"open","statement":"toki","steam":"kon","steel":"kiwen","step":"noka","stitch":"linja","stone":"kiwen","stop":"pini","story":"toki","stretch":"suli","structure":"tomo","substance":"ijo","sugar":"suwi","suggestion":"toki","summer":"tenpo","support":"awen","surprise":"nasa","swim":"tawa","system":"nasin","talk":"toki","taste":"pilin","tax":"mani","teaching":"pana","tendency":"nasin","test":"lukin","theory":"sona","thing":"ijo","thought":"pilin","time":"tenpo","tin":"kiwen","top":"sewi","touch":"luka","trade":"esun","transport":"tawa","trick":"musi","trouble":"ike","twist":"sike","unit":"wan","use":"kepeken","value":"mani","verse":"toki","vessel":"poki","view":"lukin","voice":"kalama","walk":"tawa","war":"utala","wash":"telo","waste":"jaki","water":"telo","wave":"telo","wax":"ko","way":"nasin","weather":"sewi","week":"tenpo","weight":"suli","wind":"kon","wine":"telo","winter":"tenpo","woman":"meli","wood":"kiwen","wool":"linja","word":"nimi","work":"pali","wound":"pakala","writing":"sitelen","year":"tenpo","angle":"nasin","ant":"pipi","apple":"kili","arch":"nena","arm":"luka","army":"kulupu","baby":"jan","bag":"poki","ball":"sike","band":"kulupu","basin":"poki","basket":"poki","bath":"telo","bed":"supa","bee":"pipi","bell":"ilo","berry":"kili","bird":"waso","blade":"ilo","board":"supa","boat":"tomo","bone":"kiwen","book":"lipu","boot":"len","bottle":"poki","box":"poki","boy":"mije","brain":"lawa","brake":"ilo","branch":"palisa","brick":"leko","bridge":"nasin","brush":"ilo","bucket":"poki","bulb":"ilo","button":"nena","cake":"pan","camera":"ilo","card":"lipu","carriage":"tomo","cart":"poki","cat":"soweli","chain":"linja","cheese":"ko","chest":"sinpin","chin":"uta","church":"tomo","circle":"sike","clock":"ilo","cloud":"kon","coat":"len","collar":"len","comb":"ilo","cord":"linja","cow":"mani","cup":"poki","curtain":"len","cushion":"supa","dog":"soweli","door":"lupa","drain":"weka","drawer":"poki","dress":"len","drop":"anpa","ear":"kute","egg":"sike","engine":"ilo","eye":"lukin","face":"sinpin","farm":"ma","feather":"linja","finger":"palisa","fish":"kala","flag":"len","fly":"tawa","foot":"noka","fork":"ilo","fowl":"waso","frame":"selo","garden":"ma","girl":"meli","glove":"len","goat":"soweli","gun":"ilo","hair":"linja","hammer":"ilo","hand":"luka","hat":"len","head":"lawa","heart":"pilin","hook":"ilo","horn":"palisa","horse":"soweli","hospital":"tomo","house":"tomo","island":"ma","jewel":"kiwen","kettle":"poki","key":"ilo","knee":"noka","knife":"ilo","knot":"linja","leaf":"lipu","leg":"noka","library":"tomo","line":"linja","lip":"uta","lock":"pini","map":"sitelen","match":"sama","monkey":"soweli","moon":"mun","mouth":"uta","muscle":"wawa","nail":"palisa","neck":"sijelo","needle":"palisa","nerve":"linja","net":"linja","nose":"nena","nut":"kili","office":"tomo","oven":"poki","parcel":"poki","pen":"ilo","pencil":"ilo","picture":"sitelen","pig":"soweli","pin":"palisa","pipe":"linja","plane":"tomo","plate":"supa","plow":"ilo","pocket":"poki","pot":"poki","potato":"kili","prison":"tomo","pump":"ilo","rail":"palisa","rat":"soweli","receipt":"lipu","ring":"sike","rod":"palisa","roof":"sewi","root":"noka","sail":"len","school":"tomo","scissors":"ilo","screw":"palisa","seed":"kili","sheep":"soweli","shelf":"supa","ship":"tomo","shirt":"len","shoe":"len","skin":"selo","skirt":"len","snake":"akesi","sock":"len","spade":"ilo","sponge":"ijo","spoon":"ilo","spring":"tenpo","square":"leko","stamp":"sitelen","star":"mun","station":"tomo","stick":"palisa","stocking":"len","stomach":"insa","store":"esun","street":"nasin","sun":"suno","table":"supa","tail":"linja","thread":"linja","thumb":"palisa","ticket":"lipu","toe":"palisa","tongue":"uta","tooth":"kiwen","town":"ma","train":"tomo","tray":"supa","tree":"kasi","trousers":"len","umbrella":"len","wall":"sinpin","watch":"lukin","wheel":"sike","whip":"linja","whistle":"kalama","window":"lupa","wing":"luka","wire":"linja","worm":"pipi","able":"ken","acid":"telo","angry":"pilin","automatic":"ilo","beautiful":"pona","black":"pimeja","boiling":"seli","bright":"suno","broken":"pakala","cheap":"mani","chemical":"ijo","chief":"jan","clean":"jaki","clear":"walo","common":"tenpo","complex":"ike","conscious":"sona","cut":"kipisi","deep":"mute","dependent":"wile","early":"tenpo","elastic":"linja","electric":"wawa","equal":"sama","fat":"suli","fertile":"mama","first":"nanpa","fixed":"pona","flat":"supa","free":"pona","frequent":"mute","full":"ale","general":"ale","good":"pona","great":"suli","hanging":"awen","happy":"pilin","hard":"lili","healthy":"sijelo","high":"sewi","hollow":"insa","important":"suli","kind":"pona","like":"sama","living":"lon","long":"suli","male":"mije","married":"olin","material":"ijo","medical":"misikeke","military":"kulupu","natural":"pona","necessary":"wile","new":"sin","normal":"sama","open":"open","parallel":"sama","past":"tenpo","physical":"lon","political":"lawa","poor":"ike","possible":"ken","present":"pana","private":"len","probable":"ken","quick":"wawa","quiet":"kalama","ready":"pona","red":"loje","regular":"nasin","responsible":"lawa","right":"pona","round":"sike","second":"nanpa","separate":"ante","serious":"musi","smooth":"pona","stiff":"kiwen","straight":"linja","strong":"wawa","sudden":"tenpo","sweet":"suwi","tall":"suli","thick":"suli","tight":"lili","tired":"lape","true":"lon","violent":"utala","waiting":"awen","warm":"seli","wet":"telo","wide":"suli","wise":"sona","yellow":"jelo","young":"lili","awake":"lape","bad":"ike","bent":"sike","bitter":"ike","blue":"laso","certain":"lon","cold":"lete","complete":"pini","cruel":"ike","dark":"pimeja","dead":"moli","delicate":"wawa","different":"ante","dirty":"jaki","dry":"telo","false":"lon","feeble":"wawa","female":"meli","foolish":"nasa","future":"tenpo","green":"laso","ill":"pakala","last":"nanpa","late":"tenpo","left":"awen","loose":"awen","loud":"kalama","low":"anpa","mixed":"wan","narrow":"lili","old":"sin","opposite":"ante","public":"kulupu","rough":"ike","sad":"pilin","safe":"awen","short":"lili","shut":"pini","simple":"pona","slow":"tawa","small":"lili","soft":"suwi","solid":"kiwen","special":"namako","strange":"nasa","thin":"lili","white":"walo","wrong":"ike","eh":"a","gasp":"a","gosh":"a","ha":"a","ooh":"a","oy":"a","sigh":"a","wow":"a","amphibian":"akesi","frog":"akesi","reptile":"akesi","scaly":"akesi","none":"ala","nothing":"ala","zero":"ala","attempt":"alasa","catch":"alasa","chase":"alasa","collect":"alasa","explore":"alasa","finding":"alasa","forage":"alasa","gather":"alasa","hunt":"alasa","hunting":"alasa","locate":"alasa","pursue":"alasa","pursuit":"alasa","quest":"alasa","search":"alasa","absolute":"ale","altogether":"ale","completely":"ale","comprehensive":"ale","entirely":"ale","entirety":"ale","everything":"ale","thoroughly":"ale","total":"ale","totally":"ale","universal":"ale","below":"anpa","beneath":"anpa","bottom":"anpa","bow":"anpa","defeat":"anpa","defeated":"anpa","humble":"anpa","lower":"anpa","lowly":"anpa","underlying":"anpa","underside":"anpa","adapt":"ante","adjust":"ante","affect":"ante","alter":"ante","altered":"ante","alternative":"ante","amendment":"ante","changing":"ante","contrast":"ante","conversion":"ante","convert":"ante","counterpart":"ante","difference":"ante","differently":"ante","distinct":"ante","distinction":"ante","distinguish":"ante","diversity":"ante","edit":"ante","else":"ante","modify":"ante","replace":"ante","shift":"ante","switch":"ante","transform":"ante","transformation":"ante","transition":"ante","unlike":"ante","variation":"ante","vary":"ante","decide":"anu","either":"anu","whether":"anu","disgrace":"apeja","dishonor":"apeja","embarrass":"apeja","embarrassed":"apeja","expose":"apeja","guilt":"apeja","shun":"apeja","stigma":"apeja","await":"awen","conservation":"awen","continue":"awen","continued":"awen","continuing":"awen","continuous":"awen","custody":"awen","defence":"awen","defend":"awen","defensive":"awen","delay":"awen","endure":"awen","guard":"awen","hang":"awen","hesitate":"awen","maintain":"awen","maintenance":"awen","ongoing":"awen","preserve":"awen","protect":"awen","protection":"awen","protective":"awen","remain":"awen","remaining":"awen","rescue":"awen","reservation":"awen","reserve":"awen","retain":"awen","safely":"awen","safety":"awen","save":"awen","saving":"awen","secure":"awen","security":"awen","stability":"awen","stable":"awen","stand":"awen","standing":"awen","stay":"awen","steady":"awen","suspend":"awen","sustain":"awen","wait":"awen","plus":"en","epic":"epiku","barter":"esun","bazaar":"esun","buy":"esun","commerce":"esun","commercial":"esun","deal":"esun","economy":"esun","enterprise":"esun","purchase":"esun","retail":"esun","sale":"esun","sales":"esun","sell":"esun","shop":"esun","shopping":"esun","swap":"esun","trading":"esun","transaction":"esun","anything":"ijo","asset":"ijo","element":"ijo","product":"ijo","stock":"ijo","stuff":"ijo","subject":"ijo","adj":"ike","awful":"ike","awkward":"ike","badly":"ike","challenging":"ike","chaos":"ike","complexity":"ike","complicated":"ike","corrupt":"ike","corruption":"ike","cursed":"ike","difficult":"ike","difficulty":"ike","disorder":"ike","evil":"ike","harmful":"ike","harsh":"ike","horrible":"ike","inappropriate":"ike","issue":"ike","mean":"ike","negative":"ike","offensive":"ike","poorly":"ike","problem":"ike","terrible":"ike","tragic":"ike","unfair":"ike","unfortunate":"ike","unneeded":"ike","unpleasant":"ike","vice":"ike","app":"ilo","application":"ilo","computer":"ilo","device":"ilo","equipment":"ilo","gear":"ilo","hardware":"ilo","mechanism":"ilo","robot":"ilo","software":"ilo","tech":"ilo","technical":"ilo","technological":"ilo","technology":"ilo","tool":"ilo","utility":"ilo","belly":"insa","center":"insa","central":"insa","centre":"insa","content":"insa","core":"insa","guts":"insa","inner":"insa","inside":"insa","interior":"insa","internal":"insa","within":"insa","brainstorm":"isipin","ponder":"isipin","rationalize":"isipin","bacteria":"jaki","bullshit":"jaki","disgusting":"jaki","garbage":"jaki","gross":"jaki","illness":"jaki","infection":"jaki","junk":"jaki","mess":"jaki","pollution":"jaki","repulsive":"jaki","rotten":"jaki","shit":"jaki","trash":"jaki","unclean":"jaki","unsanitary":"jaki","virus":"jaki","eliciting":"jami","sensory":"jami","stimulating":"jami","yummy":"jami","anybody":"jan","anyone":"jan","character":"jan","citizen":"jan","civilian":"jan","dude":"jan","human":"jan","individual":"jan","participant":"jan","people":"jan","personal":"jan","somebody":"jan","whoever":"jan","duplicate":"jasima","echo":"jasima","mirror":"jasima","reflect":"jasima","reverse":"jasima","amber":"jelo","golden":"jelo","yellowish":"jelo","carry":"jo","comprise":"jo","consist":"jo","contain":"jo","hold":"jo","include":"jo","own":"jo","possess":"jo","possession":"jo","marine":"kala","salmon":"kala","swimming":"kala","bang":"kalama","beep":"kalama","clap":"kalama","drum":"kalama","laugh":"kalama","sing":"kalama","tone":"kalama","arrival":"kama","arrive":"kama","arriving":"kama","become":"kama","coming":"kama","developmental":"kama","emerge":"kama","emerging":"kama","happen":"kama","induce":"kama","next":"kama","succeed":"kama","summoned":"kama","trigger":"kama","upcoming":"kama","visit":"kama","anarchy":"kamalawala","rebellion":"kamalawala","revolt":"kamalawala","uprising":"kamalawala","beige":"kapesi","bush":"kasi","herb":"kasi","vegetation":"kasi","ability":"ken","allow":"ken","available":"ken","can":"ken","capability":"ken","capable":"ken","could":"ken","eligible":"ken","enable":"ken","freedom":"ken","liberty":"ken","likelihood":"ken","likely":"ken","maybe":"ken","odds":"ken","opportunity":"ken","option":"ken","perhaps":"ken","permission":"ken","possibility":"ken","possibly":"ken","potential":"ken","potentially":"ken","privilege":"ken","probability":"ken","prospect":"ken","viable":"ken","employ":"kepeken","means":"kepeken","utilize":"kepeken","via":"kepeken","kinkajou":"kijetesantakalu","musteloid":"kijetesantakalu","raccoon":"kijetesantakalu","spiky":"kiki","triangular":"kiki","bean":"kili","kumquat":"kili","onion":"kili","tomato":"kili","vegetable":"kili","also":"kin","too":"kin","carve":"kipisi","chop":"kipisi","chunk":"kipisi","half":"kipisi","pointy":"kipisi","section":"kipisi","concrete":"kiwen","firm":"kiwen","plastic":"kiwen","rock":"kiwen","dough":"ko","goo":"ko","moldable":"ko","mud":"ko","semisolid":"ko","abstract":"kon","agent":"kon","atmosphere":"kon","breathe":"kon","breathing":"kon","breeze":"kon","definition":"kon","essence":"kon","gas":"kon","ghost":"kon","intangible":"kon","invisible":"kon","meaning":"kon","oxygen":"kon","soul":"kon","spirit":"kon","unseen":"kon","animacy":"konwe","animate":"konwe","autonomous":"konwe","autonomy":"konwe","dynamic":"konwe","organism":"konwe","dictionary":"ku","colorful":"kule","colour":"kule","diverse":"kule","flavor":"kule","genre":"kule","lgbtq":"kule","pigment":"kule","spectrum":"kule","affiliation":"kulupu","array":"kulupu","assembly":"kulupu","association":"kulupu","bunch":"kulupu","bureau":"kulupu","class":"kulupu","club":"kulupu","cluster":"kulupu","coalition":"kulupu","collection":"kulupu","collective":"kulupu","community":"kulupu","conference":"kulupu","crew":"kulupu","crowd":"kulupu","demographic":"kulupu","ethnic":"kulupu","gang":"kulupu","gathering":"kulupu","institution":"kulupu","league":"kulupu","partnership":"kulupu","rally":"kulupu","set":"kulupu","social":"kulupu","socially":"kulupu","squad":"kulupu","stack":"kulupu","team":"kulupu","tribal":"kulupu","tribe":"kulupu","hear":"kute","listen":"kute","obey":"kute","pay":"kute","depending":"la","grab":"lanpan","seize":"lanpan","steal":"lanpan","asleep":"lape","lay":"lape","cyan":"laso","indigo":"laso","teal":"laso","turquoise":"laso","administration":"lawa","administrative":"lawa","conduct":"lawa","dominate":"lawa","drive":"lawa","enforce":"lawa","executive":"lawa","govern":"lawa","institutional":"lawa","judicial":"lawa","jurisdiction":"lawa","leader":"lawa","leadership":"lawa","leading":"lawa","legal":"lawa","legislation":"lawa","main":"lawa","management":"lawa","mandate":"lawa","official":"lawa","oversee":"lawa","psychological":"lawa","regime":"lawa","regulate":"lawa","regulation":"lawa","regulatory":"lawa","ruling":"lawa","blocky":"leko","bricks":"leko","cube":"leko","polyhedron":"leko","stair":"leko","stairs":"leko","blanket":"len","clothes":"len","clothing":"len","costume":"len","covered":"len","fabric":"len","hidden":"len","hide":"len","jacket":"len","outfit":"len","privacy":"len","secret":"len","suit":"len","textile":"len","underwear":"len","wear":"len","wrap":"len","chill":"lete","freeze":"lete","frozen":"lete","raw":"lete","uncooked":"lete","barely":"lili","brief":"lili","decrease":"lili","diminish":"lili","few":"lili","fewer":"lili","gram":"lili","inch":"lili","junior":"lili","kinda":"lili","less":"lili","lightly":"lili","mild":"lili","minimal":"lili","minimize":"lili","minor":"lili","ounce":"lili","partial":"lili","partially":"lili","partly":"lili","quarter":"lili","reduce":"lili","reduction":"lili","shallow":"lili","shortage":"lili","slight":"lili","slightly":"lili","somewhat":"lili","subtle":"lili","cable":"linja","connect":"linja","fibre":"linja","flexible":"linja","fur":"linja","link":"linja","rope":"linja","row":"linja","strand":"linja","streak":"linja","string":"linja","yarn":"linja","bonded":"linluwi","bonds":"linluwi","braid":"linluwi","interlace":"linluwi","internet":"linluwi","stronger":"linluwi","weave":"linluwi","article":"lipu","bendable":"lipu","blog":"lipu","certificate":"lipu","doc":"lipu","document":"lipu","essay":"lipu","file":"lipu","journal":"lipu","literature":"lipu","magazine":"lipu","mail":"lipu","newspaper":"lipu","novel":"lipu","publication":"lipu","sheet":"lipu","tag":"lipu","website":"lipu","magenta":"loje","pink":"loje","reddish":"loje","scarlet":"loje","accurate":"lon","actual":"lon","attendance":"lon","availability":"lon","certainly":"lon","definitely":"lon","during":"lon","exactly":"lon","exist":"lon","existing":"lon","frankly":"lon","genuine":"lon","honestly":"lon","legitimate":"lon","live":"lon","located":"lon","occupy":"lon","placement":"lon","precisely":"lon","presence":"lon","real":"lon","reality":"lon","status":"lon","truly":"lon","truth":"lon","upon":"lon","yeah":"lon","yep":"lon","elbow":"luka","five":"luka","gesture":"luka","grasping":"luka","handful":"luka","limb":"luka","palm":"luka","reach":"luka","tactile":"luka","tap":"luka","wrist":"luka","apparent":"lukin","appearance":"lukin","assess":"lukin","check":"lukin","detect":"lukin","encounter":"lukin","examination":"lukin","gaze":"lukin","glance":"lukin","inspection":"lukin","investigation":"lukin","monitor":"lukin","notice":"lukin","observe":"lukin","regard":"lukin","scan":"lukin","sight":"lukin","stare":"lukin","surveillance":"lukin","visible":"lukin","cave":"lupa","dig":"lupa","doorway":"lupa","entrance":"lupa","gap":"lupa","gate":"lupa","pit":"lupa","portal":"lupa","acre":"ma","continent":"ma","dirt":"ma","domain":"ma","environmental":"ma","ground":"ma","habitat":"ma","kingdom":"ma","landscape":"ma","national":"ma","nature":"ma","outdoors":"ma","outside":"ma","province":"ma","realm":"ma","region":"ma","setting":"ma","site":"ma","territory":"ma","world":"ma","yard":"ma","zone":"ma","aged":"majuna","ancient":"majuna","elderly":"majuna","historical":"majuna","senior":"majuna","ancestor":"mama","caretaker":"mama","creator":"mama","dad":"mama","daddy":"mama","foster":"mama","founder":"mama","guardian":"mama","mama":"mama","mom":"mama","mommy":"mama","originator":"mama","parent":"mama","parental":"mama","sustainer":"mama","budget":"mani","cash":"mani","cattle":"mani","compensation":"mani","cost":"mani","currency":"mani","dollar":"mani","earnings":"mani","economic":"mani","expense":"mani","fee":"mani","fiscal":"mani","fund":"mani","funding":"mani","gold":"mani","income":"mani","investment":"mani","livestock":"mani","monetary":"mani","pension":"mani","revenue":"mani","salary":"mani","subsidy":"mani","toll":"mani","treasure":"mani","valuable":"mani","wage":"mani","wealth":"mani","wealthy":"mani","worth":"mani","feminine":"meli","girlfriend":"meli","lady":"meli","ma":"meli","wife":"meli","sapphic":"melome","wlw":"melome","average":"meso","intermediate":"meso","mediocre":"meso","medium":"meso","moderate":"meso","ordinary":"meso","me":"mi","my":"mi","myself":"mi","our":"mi","ours":"mi","ourselves":"mi","us":"mi","we":"mi","boyfriend":"mije","gentleman":"mije","husband":"mije","masculine":"mije","sir":"mije","achillean":"mijomi","mlm":"mijomi","eulipotyphla":"misa","glires":"misa","rodent":"misa","squirrel":"misa","suli":"misa","bandages":"misikeke","cure":"misikeke","exercise":"misikeke","medication":"misikeke","medicine":"misikeke","meditation":"misikeke","pill":"misikeke","prescriptions":"misikeke","therapy":"misikeke","absorb":"moku","consume":"moku","consumption":"moku","dining":"moku","dinner":"moku","drinking":"moku","eat":"moku","eating":"moku","edible":"moku","groceries":"moku","ingest":"moku","lunch":"moku","nutrition":"moku","swallow":"moku","deadly":"moli","die":"moli","dying":"moli","kill":"moli","killing":"moli","murder":"moli","ass":"monsi","behind":"monsi","butt":"monsi","rear":"monsi","afraid":"monsuta","demon":"monsuta","disturbing":"monsuta","dread":"monsuta","frightening":"monsuta","horror":"monsuta","monster":"monsuta","nervousness":"monsuta","predator":"monsuta","scare":"monsuta","scary":"monsuta","terror":"monsuta","threat":"monsuta","neigh":"mu","purr":"mu","ribbit":"mu","vocalization":"mu","woof":"mu","celestial":"mun","amusing":"musi","artistic":"musi","comedy":"musi","comical":"musi","enjoy":"musi","entertainment":"musi","fun":"musi","funny":"musi","game":"musi","humour":"musi","interesting":"musi","irony":"musi","playoffs":"musi","sport":"musi","billion":"mute","dose":"mute","dozen":"mute","eight":"mute","exceed":"mute","excessive":"mute","extremely":"mute","fifteen":"mute","fifty":"mute","increasingly":"mute","incredibly":"mute","lots":"mute","majority":"mute","many":"mute","million":"mute","most":"mute","mostly":"mute","nine":"mute","numerous":"mute","particularly":"mute","plenty":"mute","several":"mute","six":"mute","ten":"mute","thirty":"mute","thousand":"mute","trillion":"mute","twelve":"mute","twenty":"mute","various":"mute","er":"n","hm":"n","mm":"n","mmm":"n","umm":"n","additional":"namako","adornment":"namako","bonus":"namako","extra":"namako","fancy":"namako","ornament":"namako","spice":"namako","calculate":"nanpa","count":"nanpa","edition":"nanpa","grade":"nanpa","measurement":"nanpa","percent":"nanpa","rank":"nanpa","ranking":"nanpa","rating":"nanpa","score":"nanpa","statistical":"nanpa","statistics":"nanpa","th":"nanpa","version":"nanpa","abnormal":"nasa","confuse":"nasa","drunk":"nasa","intoxicated":"nasa","nonsense":"nasa","odd":"nasa","psychoactive":"nasa","ridiculous":"nasa","suspicious":"nasa","unexpected":"nasa","unusual":"nasa","weird":"nasa","wild":"nasa","aisle":"nasin","avenue":"nasin","channel":"nasin","composition":"nasin","custom":"nasin","doctrine":"nasin","format":"nasin","guideline":"nasin","ideology":"nasin","lane":"nasin","layout":"nasin","manner":"nasin","method":"nasin","mode":"nasin","norm":"nasin","orientation":"nasin","passage":"nasin","path":"nasin","protocol":"nasin","route":"nasin","routine":"nasin","strategy":"nasin","style":"nasin","tactic":"nasin","technique":"nasin","theme":"nasin","track":"nasin","tradition":"nasin","trail":"nasin","bump":"nena","hill":"nena","protuberances":"nena","ridge":"nena","specifically":"ni","these":"ni","those":"ni","brand":"nimi","label":"nimi","lyrics":"nimi","term":"nimi","title":"nimi","joke":"nimisin","ankle":"noka","heel":"noka","lap":"noka","locomotion":"noka","roots":"noka","english":"ojuta","ligma":"ojuta","meme":"ojuta","uta":"ojuta","affection":"olin","appreciation":"olin","beloved":"olin","compassion":"olin","faithful":"olin","familial":"olin","intimate":"olin","loving":"olin","loyal":"olin","loyalty":"olin","passion":"olin","platonic":"olin","relationships":"olin","respect":"olin","romance":"olin","romantic":"olin","farewell":"omekapo","goodbye":"omekapo","kala":"omekapo","later":"omekapo","moku":"omekapo","her":"ona","herself":"ona","him":"ona","himself":"ona","his":"ona","itself":"ona","she":"ona","themselves":"ona","they":"ona","access":"open","begin":"open","beginning":"open","engage":"open","entry":"open","initial":"open","initiate":"open","introduction":"open","launch":"open","starter":"open","starting":"open","nineteen":"owe","orwellian":"owe","relating":"owe","totalitarian":"owe","accident":"pakala","boom":"pakala","botched":"pakala","collapse":"pakala","crash":"pakala","damaged":"pakala","damn":"pakala","destroy":"pakala","explode":"pakala","fail":"pakala","failure":"pakala","fault":"pakala","flaw":"pakala","goddamn":"pakala","harmed":"pakala","heck":"pakala","hurt":"pakala","injure":"pakala","injury":"pakala","messed":"pakala","mistake":"pakala","offense":"pakala","ruin":"pakala","snap":"pakala","violate":"pakala","forbid":"pake","halt":"pake","accomplish":"pali","accomplishment":"pali","achieve":"pali","action":"pali","actively":"pali","assemble":"pali","assignment":"pali","behaviour":"pali","behavioural":"pali","build":"pali","busy":"pali","career":"pali","commit":"pali","construct":"pali","craft":"pali","create":"pali","creation":"pali","design":"pali","develop":"pali","effort":"pali","employment":"pali","enact":"pali","execute":"pali","function":"pali","functional":"pali","generate":"pali","implementation":"pali","industrial":"pali","invent":"pali","job":"pali","labour":"pali","manipulate":"pali","manufacturing":"pali","occupation":"pali","operating":"pali","participation":"pali","perform":"pali","prepare":"pali","processing":"pali","production":"pali","productive":"pali","profession":"pali","project":"pali","role":"pali","service":"pali","task":"pali","undergo":"pali","undertake":"pali","working":"pali","bar":"palisa","mast":"palisa","pole":"palisa","spine":"palisa","staff":"palisa","stake":"palisa","baked":"pan","carbohydrate":"pan","corn":"pan","foods":"pan","goods":"pan","grains":"pan","injera":"pan","masa":"pan","noodles":"pan","porridge":"pan","sorghum":"pan","starchy":"pan","wheat":"pan","add":"pana","added":"pana","apply":"pana","appoint":"pana","assign":"pana","contribute":"pana","contribution":"pana","convey":"pana","dedicate":"pana","deliver":"pana","delivery":"pana","deploy":"pana","deposit":"pana","devote":"pana","distribute":"pana","donate":"pana","donation":"pana","emission":"pana","emit":"pana","exhibit":"pana","export":"pana","gift":"pana","given":"pana","grant":"pana","input":"pana","install":"pana","introduce":"pana","lend":"pana","offering":"pana","output":"pana","post":"pana","pour":"pana","provide":"pana","provision":"pana","publish":"pana","publishing":"pana","release":"pana","serve":"pana","serving":"pana","share":"pana","shared":"pana","show":"pana","spend":"pana","spread":"pana","submit":"pana","supply":"pana","toss":"pana","transmission":"pana","yield":"pana","taso":"penpo","concerning":"pi","conductive":"pika","electronic":"pika","lightning":"pika","online":"pika","assume":"pilin","assumption":"pilin","attitude":"pilin","believe":"pilin","conscience":"pilin","consider":"pilin","deem":"pilin","emotion":"pilin","emotionally":"pilin","estimated":"pilin","hypothesis":"pilin","mentally":"pilin","mood":"pilin","notion":"pilin","outlook":"pilin","sentiment":"pilin","speculation":"pilin","stimulus":"pilin","suppose":"pilin","suspicion":"pilin","darkness":"pimeja","shadow":"pimeja","unlit":"pimeja","cancel":"pini","close":"pini","closed":"pini","conclusion":"pini","disable":"pini","ended":"pini","ending":"pini","final":"pini","finish":"pini","latter":"pini","outcome":"pini","quit":"pini","result":"pini","bug":"pipi","spider":"pipi","four":"po","alongside":"poka","aside":"poka","beside":"poka","closely":"poka","hip":"poka","nearby":"poka","sidebar":"poka","surroundings":"poka","vicinity":"poka","bowl":"poki","cabinet":"poki","cage":"poki","case":"poki","cell":"poki","classify":"poki","container":"poki","cupboard":"poki","folder":"poki","jar":"poki","locker":"poki","package":"poki","pitcher":"poki","purse":"poki","sack":"poki","storage":"poki","trunk":"poki","acceptable":"pona","adequate":"pona","advantage":"pona","alright":"pona","basic":"pona","benefit":"pona","better":"pona","civil":"pona","comfortable":"pona","correction":"pona","courtesy":"pona","decent":"pona","ease":"pona","easily":"pona","easy":"pona","effective":"pona","effectiveness":"pona","efficiency":"pona","elegant":"pona","elementary":"pona","enhance":"pona","ethical":"pona","excellent":"pona","fairly":"pona","fix":"pona","friendly":"pona","grace":"pona","heal":"pona","helpful":"pona","honest":"pona","ideal":"pona","improve":"pona","improved":"pona","improvement":"pona","justice":"pona","lovely":"pona","mercy":"pona","merit":"pona","morality":"pona","nice":"pona","ok":"pona","peaceful":"pona","perfect":"pona","plain":"pona","pleasant":"pona","proper":"pona","properly":"pona","qualify":"pona","reasonable":"pona","reliability":"pona","relieve":"pona","repair":"pona","resolve":"pona","simply":"pona","solve":"pona","success":"pona","successful":"pona","successfully":"pona","sufficient":"pona","supportive":"pona","terrific":"pona","thanks":"pona","useful":"pona","valid":"pona","virtue":"pona","welfare":"pona","worthy":"pona","deceive":"powe","fake":"powe","fraud":"powe","myth":"powe","pretend":"powe","saying":"powe","unreal":"powe","untrue":"powe","untruth":"powe","officially":"pu","fluffy":"puwa","alike":"sama","associated":"sama","equality":"sama","equally":"sama","equity":"sama","equivalent":"sama","fellow":"sama","identical":"sama","likewise":"sama","mutual":"sama","peer":"sama","relate":"sama","related":"sama","resemble":"sama","same":"sama","similar":"sama","similarly":"sama","three":"san","bake":"seli","boil":"seli","burning":"seli","cooking":"seli","hot":"seli","temperature":"seli","warming":"seli","bark":"selo","border":"selo","boundary":"selo","external":"selo","layer":"selo","margin":"selo","outer":"selo","outline":"selo","peel":"selo","shell":"selo","what":"seme","above":"sewi","divine":"sewi","elevated":"sewi","excelling":"sewi","god":"sewi","highest":"sewi","holy":"sewi","inspiring":"sewi","raise":"sewi","religious":"sewi","rise":"sewi","sacred":"sewi","spiritual":"sewi","summit":"sewi","superior":"sewi","supreme":"sewi","upper":"sewi","upstairs":"sewi","figure":"sijelo","flesh":"sijelo","torso":"sijelo","circuit":"sike","cycle":"sike","disc":"sike","globe":"sike","loop":"sike","orbit":"sike","repeating":"sike","rolling":"sike","sphere":"sike","spin":"sike","spiral":"sike","spot":"sike","surround":"sike","fresh":"sin","innovation":"sin","innovative":"sin","modern":"sin","newly":"sin","recent":"sin","repeat":"sin","returned":"sin","unprecedented":"sin","update":"sin","yours":"sina","yourself":"sina","ahead":"sinpin","vertical":"sinpin","alphabet":"sitelen","depict":"sitelen","draw":"sitelen","drawing":"sitelen","graph":"sitelen","graphic":"sitelen","icon":"sitelen","illustrate":"sitelen","illustration":"sitelen","image":"sitelen","painting":"sitelen","photo":"sitelen","photograph":"sitelen","portrait":"sitelen","portray":"sitelen","render":"sitelen","representation":"sitelen","symbol":"sitelen","write":"sitelen","lichen":"soko","about":"sona","acknowledge":"sona","aware":"sona","awareness":"sona","clever":"sona","data":"sona","educational":"sona","experienced":"sona","expertise":"sona","familiar":"sona","how":"sona","identify":"sona","info":"sona","information":"sona","insight":"sona","intelligence":"sona","know":"sona","known":"sona","logic":"sona","logical":"sona","recognition":"sona","recognize":"sona","remember":"sona","skill":"sona","skilled":"sona","smart":"sona","understand":"sona","understanding":"sona","wisdom":"sona","port":"soto","deer":"soweli","fuzzy":"soweli","lion":"soweli","llama":"soweli","tiger":"soweli","wolf":"soweli","began":"su","illustrated":"su","oz":"su","produced":"su","series":"su","wizard":"su","age":"suli","broad":"suli","considerable":"suli","critical":"suli","crucial":"suli","density":"suli","depth":"suli","dominant":"suli","dramatic":"suli","dramatically":"suli","emphasis":"suli","emphasize":"suli","enormous":"suli","essential":"suli","expand":"suli","extend":"suli","extended":"suli","extensive":"suli","extent":"suli","fundamental":"suli","giant":"suli","grand":"suli","grow":"suli","heavily":"suli","heavy":"suli","height":"suli","huge":"suli","importance":"suli","importantly":"suli","influential":"suli","largely":"suli","length":"suli","magnitude":"suli","mainly":"suli","major":"suli","massive":"suli","profound":"suli","prominent":"suli","relevant":"suli","remarkable":"suli","significance":"suli","significant":"suli","significantly":"suli","substantial":"suli","super":"suli","tremendous":"suli","vast":"suli","vital":"suli","volume":"suli","widely":"suli","brightness":"suno","glow":"suno","radiance":"suno","shine":"suno","solar":"suno","sunlight":"suno","bench":"supa","chair":"supa","couch":"supa","counter":"supa","deck":"supa","desk":"supa","floor":"supa","furniture":"supa","horizontal":"supa","platform":"supa","platypus":"sutopatikuna","adorable":"suwi","candy":"suwi","cute":"suwi","darling":"suwi","dessert":"suwi","fragrant":"suwi","gentle":"suwi","tender":"suwi","treat":"suwi","attract":"taki","magnetic":"taki","basis":"tan","because":"tan","behalf":"tan","factor":"tan","per":"tan","since":"tan","although":"taso","despite":"taso","except":"taso","exception":"taso","exclusive":"taso","exclusively":"taso","however":"taso","just":"taso","merely":"taso","nevertheless":"taso","nonetheless":"taso","regardless":"taso","sole":"taso","solely":"taso","whereas":"taso","yet":"taso","according":"tawa","advance":"tawa","approach":"tawa","cross":"tawa","departure":"tawa","drift":"tawa","forth":"tawa","going":"tawa","march":"tawa","migration":"tawa","mobile":"tawa","momentum":"tawa","movement":"tawa","pass":"tawa","perspective":"tawa","proceed":"tawa","progress":"tawa","shaking":"tawa","slide":"tawa","towards":"tawa","transit":"tawa","transportation":"tawa","travel":"tawa","trip":"tawa","until":"tawa","walking":"tawa","starboard":"teje","fluid":"telo","gasoline":"telo","ink":"telo","lake":"telo","lava":"telo","liquids":"telo","oil":"telo","pond":"telo","pool":"telo","shower":"telo","soda":"telo","appointment":"tenpo","circumstance":"tenpo","date":"tenpo","duration":"tenpo","instance":"tenpo","interval":"tenpo","moment":"tenpo","occasion":"tenpo","period":"tenpo","phase":"tenpo","scenario":"tenpo","season":"tenpo","session":"tenpo","situation":"tenpo","timing":"tenpo","admit":"toki","call":"toki","claim":"toki","comment":"toki","commentary":"toki","communicate":"toki","confess":"toki","consult":"toki","contact":"toki","conversation":"toki","counsel":"toki","declaration":"toki","declare":"toki","describe":"toki","dialogue":"toki","disclose":"toki","discourse":"toki","discuss":"toki","excuse":"toki","express":"toki","expression":"toki","feedback":"toki","greet":"toki","hello":"toki","hi":"toki","interaction":"toki","interview":"toki","mention":"toki","message":"toki","narrative":"toki","negotiate":"toki","preach":"toki","proposal":"toki","proposition":"toki","quote":"toki","refer":"toki","remark":"toki","reply":"toki","report":"toki","reporting":"toki","respond":"toki","response":"toki","specify":"toki","speech":"toki","suggest":"toki","tale":"toki","tell":"toki","testify":"toki","verbal":"toki","apartment":"tomo","cabin":"tomo","chamber":"tomo","domestic":"tomo","facility":"tomo","hall":"tomo","home":"tomo","household":"tomo","housing":"tomo","indoor":"tomo","institute":"tomo","residence":"tomo","residential":"tomo","shack":"tomo","shelter":"tomo","space":"tomo","structural":"tomo","suite":"tomo","tent":"tomo","gender":"tonsi","genderqueer":"tonsi","intersex":"tonsi","nonbinary":"tonsi","nonconforming":"tonsi","couple":"tu","double":"tu","pair":"tu","twice":"tu","two":"tu","sexual":"unpa","sexually":"unpa","violet":"unu","enchant":"usawi","incomprehensible":"usawi","magic":"usawi","occult":"usawi","sorcery":"usawi","consuming":"uta","jaw":"uta","lips":"uta","throat":"uta","aggressive":"utala","assault":"utala","battle":"utala","beat":"utala","challenge":"utala","combat":"utala","compete":"utala","competitive":"utala","conflict":"utala","confront":"utala","contend":"utala","contest":"utala","dispute":"utala","fighting":"utala","hit":"utala","oppose":"utala","opposition":"utala","punch":"utala","resist":"utala","riot":"utala","struggle":"utala","tackle":"utala","versus":"utala","violence":"utala","pale":"walo","alone":"wan","bind":"wan","blend":"wan","combine":"wan","combined":"wan","component":"wan","fuse":"wan","independent":"wan","integrate":"wan","integration":"wan","join":"wan","mix":"wan","primary":"wan","solo":"wan","union":"wan","unite":"wan","united":"wan","unity":"wan","bat":"waso","chicken":"waso","duck":"waso","eagle":"waso","flying":"waso","turkey":"waso","winged":"waso","qualities":"wasoweli","soweli":"wasoweli","waso":"wasoweli","bold":"wawa","boost":"wawa","brave":"wawa","charge":"wawa","compelling":"wawa","confident":"wawa","electrical":"wawa","energy":"wawa","fast":"wawa","firmly":"wawa","forceful":"wawa","glory":"wawa","impressive":"wawa","intense":"wawa","intensity":"wawa","powerful":"wawa","quickly":"wawa","rapid":"wawa","rapidly":"wawa","reinforce":"wawa","strength":"wawa","strengthen":"wawa","strongly":"wawa","tough":"wawa","abandon":"weka","absence":"weka","apart":"weka","avoid":"weka","away":"weka","ban":"weka","delete":"weka","disappear":"weka","dismiss":"weka","distant":"weka","eliminate":"weka","escape":"weka","exclude":"weka","exit":"weka","fade":"weka","flee":"weka","lack":"weka","lose":"weka","missing":"weka","omit":"weka","reject":"weka","remote":"weka","removal":"weka","remove":"weka","resign":"weka","rid":"weka","skip":"weka","strip":"weka","sweep":"weka","vanish":"weka","wipe":"weka","withdraw":"weka","returning":"wekama","temporarily":"wekama","agenda":"wile","ambition":"wile","choose":"wile","consent":"wile","criteria":"wile","dare":"wile","demand":"wile","depend":"wile","deserve":"wile","determination":"wile","eager":"wile","elect":"wile","enthusiasm":"wile","goal":"wile","intend":"wile","intent":"wile","intention":"wile","interested":"wile","motivation":"wile","necessity":"wile","objective":"wile","obligation":"wile","opt":"wile","pick":"wile","prefer":"wile","preference":"wile","require":"wile","required":"wile","requirement":"wile","supposed":"wile","urge":"wile","want":"wile","willing":"wile","willingness":"wile","would":"wile","yearn":"wile","banned":"wuwojiti","ji":"wuwojiti","phonotactics":"wuwojiti","syllables":"wuwojiti","ti":"wuwojiti","wo":"wuwojiti","wu":"wuwojiti","actually":"yupekosi","george":"yupekosi","lucas":"yupekosi","revise":"yupekosi","unintentionally":"yupekosi","works":"yupekosi","worse":"yupekosi"},"irregular":{"went":"go","found":"find","saw":"see","came":"come","ran":"run","ate":"eat","made":"make","took":"take","gave":"give","said":"say","told":"tell","felt":"feel","thought":"think","knew":"know","got":"get","left":"leave","sat":"sit","stood":"stand","fell":"fall","flew":"fly","swam":"swim","sang":"sing","drank":"drink","slept":"sleep","heard":"hear","held":"hold","built":"build","bought":"buy","brought":"bring","began":"begin","became":"become","wrote":"write","grew":"grow","threw":"throw","kept":"keep","met":"meet","lost":"lose","children":"child","people":"person","men":"man","women":"woman","feet":"foot","teeth":"tooth","mice":"mouse"}}')  # Basic English word -> toki pona hub (experiments/basic-english)


# HUBS["table"]: English word -> toki pona hub (the 850 Basic English words mapped earlier, plus words whose
# Linku gloss points at exactly one toki pona word). HUBS["irregular"]: went -> go and so on.
def lemma(w):
    T, IRR = HUBS["table"], HUBS["irregular"]
    if w in T:
        return w
    if w in IRR and IRR[w] in T:
        return IRR[w]
    for suf, rep in (("ies", "y"), ("es", ""), ("s", ""), ("ied", "y"), ("ed", ""), ("ed", "e"), ("ing", ""), ("ing", "e"), ("ly", "")):
        if w.endswith(suf):
            b = w[: -len(suf)] + rep
            if b in T:
                return b
            if len(b) > 2 and b[-1] == b[-2] and b[:-1] in T:
                return b[:-1]
    return None


def mix(story, r, p=0.5):
    """Swap English words for their toki pona hub word, half the time (about 45% of words have a hub)."""
    def sw(m):
        w = m.group(0)
        b = lemma(w.lower())
        return HUBS["table"][b] if b and r.random() < p else w
    return re.sub(r"[A-Za-z]+", sw, story)


r = random.Random(1)
bridge_pairs = [f"tp: {a}\nen: {b}" if r.random() < 0.5 else f"en: {b}\ntp: {a}" for a, b in pairs]
mixed_stories = [mix(s, r) for s in budget]
log(f"toki pona {len(tp_text)} texts, pairs {len(pairs)}, budget {len(budget)} stories, ceiling pool {len(ceil_pool)}")
print("mixed example:", mixed_stories[0][:300])

## Tokenizer: SentencePiece BPE, Llama-style (so it exports to GGUF / Ollama)

In [ ]:
with open("spm_train.txt", "w") as f:
    for s in tp_text + [p[1] for p in pairs] + ceil_pool[:60_000]:
        f.write(s + "\n")
spm.SentencePieceTrainer.train(
    input="spm_train.txt", model_prefix="tok", vocab_size=VOCAB, model_type="bpe", byte_fallback=True,
    character_coverage=0.9995, normalization_rule_name="identity", add_dummy_prefix=True,
    remove_extra_whitespaces=False, split_digits=True, allow_whitespace_only_pieces=True,
    pad_id=3, unk_id=0, bos_id=1, eos_id=2, input_sentence_size=400_000, shuffle_input_sentence=True,
    num_threads=os.cpu_count(), minloglevel=2)
SP = spm.SentencePieceProcessor(model_file="tok.model")
V, EOS, PAD = SP.get_piece_size(), SP.eos_id(), SP.pad_id()


def stream(texts):
    ids = []
    for chunk in SP.encode(texts):
        ids += chunk + [EOS]
    return torch.tensor(ids, dtype=torch.long)


log("tokenising")
S = {"tp": stream(tp_text), "tp_val": stream(tp_val), "bridge": stream(bridge_pairs), "mixed": stream(mixed_stories),
     "en": stream(en_budget), "val": stream(val_stories)}
S["ceil"] = stream(ceil_pool)
for k, v in S.items():
    print(f"  {k:7} {len(v)/1e6:6.2f}M tokens")

## Model and training

In [ ]:
def make_model():
    from transformers import LlamaConfig, LlamaForCausalLM
    cfg = LlamaConfig(vocab_size=V, hidden_size=D, intermediate_size=FFN, num_hidden_layers=LAYERS,
                      num_attention_heads=HEADS, num_key_value_heads=HEADS, max_position_embeddings=CTX,
                      rms_norm_eps=1e-6, rope_theta=10000.0, tie_word_embeddings=True,
                      bos_token_id=1, eos_token_id=EOS, pad_token_id=PAD)
    try:
        m = LlamaForCausalLM._from_config(cfg, attn_implementation="sdpa")
    except Exception:
        m = LlamaForCausalLM(cfg)
    return m.to(DEV)


def windows(src, n, g):
    i = torch.randint(0, len(src) - CTX - 1, (n,), generator=g)
    idx = i[:, None] + torch.arange(CTX + 1)
    return src[idx]


VAL_WIN = windows(S["val"], 256 if not QUICK else 16, torch.Generator().manual_seed(7))
TPVAL_WIN = windows(S["tp_val"], 128 if not QUICK else 16, torch.Generator().manual_seed(8))


@torch.no_grad()
def val_loss(model, win=None):
    win = VAL_WIN if win is None else win
    model.eval()
    tot = 0.0
    for b in win.split(32):
        b = b.to(DEV)
        with torch.autocast(**AMP):
            lg = model(input_ids=b[:, :-1]).logits
        tot += F.cross_entropy(lg.float().reshape(-1, V), b[:, 1:].reshape(-1), reduction="sum").item()
    model.train()
    return tot / (win.shape[0] * CTX)


def make_scaler():
    try:
        return torch.amp.GradScaler("cuda", enabled=DEV == "cuda")
    except (AttributeError, TypeError):
        return torch.cuda.amp.GradScaler(enabled=DEV == "cuda")


class Trainer:
    """One optimizer and LR schedule across all stages; tracks the best English validation loss."""
    def __init__(self, model, total):
        self.m, self.total, self.step = model, total, 0
        self.opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.1, betas=(0.9, 0.95))
        self.scaler, self.g = make_scaler(), torch.Generator().manual_seed(SEED)
        self.best, self.best_state, self.curve = float("inf"), None, []

    def run(self, mixture, steps, track=True, name=""):
        names, weights = zip(*mixture)
        t = time.time()
        self.m.train()
        for k in range(steps):
            frac = self.step / self.total
            lr = LR * min(1, (self.step + 1) / 200) * (0.1 + 0.9 * 0.5 * (1 + math.cos(math.pi * frac)))
            for gr in self.opt.param_groups:
                gr["lr"] = lr
            counts = torch.multinomial(torch.tensor(weights, dtype=torch.float), BATCH, replacement=True, generator=self.g).bincount(minlength=len(names))
            b = torch.cat([windows(S[n], int(c), self.g) for n, c in zip(names, counts) if c > 0]).to(DEV)
            with torch.autocast(**AMP):
                lg = self.m(input_ids=b[:, :-1]).logits
            loss = F.cross_entropy(lg.float().reshape(-1, V), b[:, 1:].reshape(-1))
            self.opt.zero_grad(set_to_none=True)
            self.scaler.scale(loss).backward()
            self.scaler.unscale_(self.opt)
            nn.utils.clip_grad_norm_(self.m.parameters(), 1.0)
            self.scaler.step(self.opt); self.scaler.update()
            self.step += 1
            if self.step % EVAL_EVERY == 0 or k == steps - 1:
                v = val_loss(self.m)
                self.curve.append((self.step, v))
                if track and v < self.best:
                    self.best = v
                    self.best_state = {kk: vv.detach().clone() for kk, vv in self.m.state_dict().items()}
                eta = (time.time() - t) / (k + 1) * (steps - k - 1) / 60
                log(f"  {name} step {self.step}/{self.total}  train {loss.item():.3f}  english val {v:.3f}  best {self.best:.3f}  (stage eta {eta:.0f} min)")

    def finish(self):
        if self.best_state is not None:
            self.m.load_state_dict(self.best_state)
        return self.m


def train_condition(cond):
    torch.manual_seed(SEED)
    model = make_model()
    tr = Trainer(model, S1 + S2 + S3)
    if cond == "simple-first":
        tr.run([("tp", 1.0)], S1, track=False, name=cond)
        tr.run([("bridge", 0.4), ("mixed", 0.4), ("tp", 0.2)], S2, track=False, name=cond)
        tr.run([("en", 0.9), ("tp", 0.1)], S3, name=cond)
    elif cond == "english":
        tr.run([("en", 1.0)], S1 + S2 + S3, name=cond)
    else:
        tr.run([("ceil", 1.0)], S1 + S2 + S3, name=cond)
    model = tr.finish()
    return model, tr.curve

## Tests: BLiMP and bAbI

In [ ]:
@torch.no_grad()
def seq_logprob(model, texts, prefix_lens=None):
    """Sum of log-probs of each text's tokens (after an optional prefix of n tokens)."""
    model.eval()
    out = []
    enc = [[EOS] + x for x in SP.encode(texts)]
    order = sorted(range(len(enc)), key=lambda i: len(enc[i]))
    res = [0.0] * len(enc)
    for s in range(0, len(order), 128):
        idx = order[s:s + 128]
        L = min(max(len(enc[i]) for i in idx), CTX + 1)
        x = torch.full((len(idx), L), PAD, dtype=torch.long)
        m = torch.zeros(len(idx), L - 1)
        for j, i in enumerate(idx):
            e = enc[i][-L:]
            x[j, :len(e)] = torch.tensor(e)
            # position j predicts token j+1; score every token from index prefix_len on (index 1 on with no prefix)
            start = 0 if prefix_lens is None else max(0, prefix_lens[i] - 1 - (len(enc[i]) - len(e)))
            m[j, start:len(e) - 1] = 1
        x, m = x.to(DEV), m.to(DEV)
        with torch.autocast(**AMP):
            lg = model(input_ids=x[:, :-1]).logits
        lp = F.log_softmax(lg.float(), -1).gather(-1, x[:, 1:].unsqueeze(-1)).squeeze(-1)
        for j, i in enumerate(idx):
            res[i] = (lp[j] * m[j]).sum().item()
    return res


BLIMP_CONFIGS = ['adjunct_island', 'anaphor_gender_agreement', 'anaphor_number_agreement', 'animate_subject_passive', 'animate_subject_trans', 'causative', 'complex_NP_island', 'coordinate_structure_constraint_complex_left_branch', 'coordinate_structure_constraint_object_extraction', 'determiner_noun_agreement_1', 'determiner_noun_agreement_2', 'determiner_noun_agreement_irregular_1', 'determiner_noun_agreement_irregular_2', 'determiner_noun_agreement_with_adj_2', 'determiner_noun_agreement_with_adj_irregular_1', 'determiner_noun_agreement_with_adj_irregular_2', 'determiner_noun_agreement_with_adjective_1', 'distractor_agreement_relational_noun', 'distractor_agreement_relative_clause', 'drop_argument', 'ellipsis_n_bar_1', 'ellipsis_n_bar_2', 'existential_there_object_raising', 'existential_there_quantifiers_1', 'existential_there_quantifiers_2', 'existential_there_subject_raising', 'expletive_it_object_raising', 'inchoative', 'intransitive', 'irregular_past_participle_adjectives', 'irregular_past_participle_verbs', 'irregular_plural_subject_verb_agreement_1', 'irregular_plural_subject_verb_agreement_2', 'left_branch_island_echo_question', 'left_branch_island_simple_question', 'matrix_question_npi_licensor_present', 'npi_present_1', 'npi_present_2', 'only_npi_licensor_present', 'only_npi_scope', 'passive_1', 'passive_2', 'principle_A_c_command', 'principle_A_case_1', 'principle_A_case_2', 'principle_A_domain_1', 'principle_A_domain_2', 'principle_A_domain_3', 'principle_A_reconstruction', 'regular_plural_subject_verb_agreement_1', 'regular_plural_subject_verb_agreement_2', 'sentential_negation_npi_licensor_present', 'sentential_negation_npi_scope', 'sentential_subject_island', 'superlative_quantifiers_1', 'superlative_quantifiers_2', 'tough_vs_raising_1', 'tough_vs_raising_2', 'transitive', 'wh_island', 'wh_questions_object_gap', 'wh_questions_subject_gap', 'wh_questions_subject_gap_long_distance', 'wh_vs_that_no_gap', 'wh_vs_that_no_gap_long_distance', 'wh_vs_that_with_gap', 'wh_vs_that_with_gap_long_distance']
log("loading BLiMP and bAbI")
BLIMP = []
for c in (BLIMP_CONFIGS[:4] if QUICK else BLIMP_CONFIGS):
    d = load_dataset("nyu-mll/blimp", c, split="train")
    BLIMP += [(c, r["field"], r["sentence_good"], r["sentence_bad"]) for r in d.select(range(min(BLIMP_N, len(d))))]
babi_tr = load_dataset("Muennighoff/babi", split="train")
babi_te = load_dataset("Muennighoff/babi", split="test")
BABI_ANS = {}
for r_ in babi_tr:
    BABI_ANS.setdefault(r_["task"], set()).add(r_["answer"])
BABI_TRAIN = [(f"{clean(r_['passage'])}\nQuestion: {r_['question']}\nAnswer:", " " + r_["answer"]) for r_ in babi_tr]
_bt = {}
for r_ in babi_te:
    _bt.setdefault(r_["task"], []).append(r_)
BABI_TEST = [(t, f"{clean(x['passage'])}\nQuestion: {x['question']}\nAnswer:", x["answer"]) for t, xs in _bt.items()
             for x in random.Random(t).sample(xs, min(BABI_N, len(xs)))]


def blimp(model):
    good = seq_logprob(model, [b[2] for b in BLIMP])
    bad = seq_logprob(model, [b[3] for b in BLIMP])
    by = {}
    for (c, field, _, _), g, b in zip(BLIMP, good, bad):
        by.setdefault(field, []).append(g > b)
    allv = [x for v in by.values() for x in v]
    return {"overall": st.mean(allv), **{k: st.mean(v) for k, v in by.items()}}


def babi(model):
    """Same short fine-tune for every model (answer tokens only), then pick the most likely answer."""
    import copy
    m = copy.deepcopy(model)
    m.train()
    opt = torch.optim.AdamW(m.parameters(), lr=3e-4, weight_decay=0.0)
    scaler, r = make_scaler(), random.Random(3)
    for step in range(BABI_STEPS):
        items = [r.choice(BABI_TRAIN) for _ in range(32)]
        encs = [([EOS] + SP.encode(q))[-(CTX - 8):] for q, _ in items]
        ans = [SP.encode(a.strip()) for _, a in items]
        L = max(len(e) + len(a) for e, a in zip(encs, ans))
        x = torch.full((len(items), L), PAD, dtype=torch.long)
        w = torch.zeros(len(items), L - 1)
        for j, (e, a) in enumerate(zip(encs, ans)):
            x[j, :len(e) + len(a)] = torch.tensor(e + a)
            w[j, len(e) - 1:len(e) + len(a) - 1] = 1
        x, w = x.to(DEV), w.to(DEV)
        with torch.autocast(**AMP):
            lg = m(input_ids=x[:, :-1]).logits
        loss = (F.cross_entropy(lg.float().reshape(-1, V), x[:, 1:].reshape(-1), reduction="none") * w.reshape(-1)).sum() / w.sum()
        opt.zero_grad(set_to_none=True)
        scaler.scale(loss).backward(); scaler.unscale_(opt)
        nn.utils.clip_grad_norm_(m.parameters(), 1.0)
        scaler.step(opt); scaler.update()
    texts, owners, prefix = [], [], []
    for i, (t, q, a) in enumerate(BABI_TEST):
        n = len(SP.encode(q)) + 1
        for cand in sorted(BABI_ANS[t]):
            texts.append(q + " " + cand); owners.append((i, cand)); prefix.append(n)
    lp = seq_logprob(m, texts, prefix_lens=prefix)
    best = {}
    for (i, cand), v in zip(owners, lp):
        if i not in best or v > best[i][1]:
            best[i] = (cand, v)
    per = {}
    for i, (t, q, a) in enumerate(BABI_TEST):
        per.setdefault(t, []).append(best[i][0] == a)
    del m
    return {"mean": st.mean(st.mean(v) for v in per.values()), **{f"task{t}": st.mean(v) for t, v in sorted(per.items())}}


@torch.no_grad()
def samples(model, prompts=("Once upon a time", "The little dog", "Mia wanted to")):
    model.eval()
    out = []
    for p in prompts:
        x = torch.tensor([[EOS] + SP.encode(p)], device=DEV)
        g = model.generate(x, max_new_tokens=80, do_sample=True, temperature=0.7, top_k=40, pad_token_id=PAD, eos_token_id=EOS)
        out.append(SP.decode(g[0, 1:].tolist()))
    return out

## Export: GGUF (F16) for Ollama / llama.cpp

In [ ]:
GGUF_T = {"u32": 4, "i32": 5, "f32": 6, "bool": 7, "str": 8, "arr": 9, "u64": 10}


def _gs(out, s):
    b = s.encode("utf-8")
    out += struct.pack("<Q", len(b)) + b


def _gv(out, typ, v):
    if typ == "str":
        _gs(out, v)
    else:
        out += struct.pack({"u32": "<I", "i32": "<i", "f32": "<f", "bool": "<?", "u64": "<Q"}[typ], v)


def write_gguf(path, kvs, tensors, align=32):
    """tensors: [(name, torch tensor)]; 2-D weights stored as F16, the rest as F32."""
    blobs = []
    for name, t in tensors:
        t = t.detach().float().cpu().contiguous()
        if t.dim() == 2:
            blobs.append((name, list(t.shape), 1, t.half().numpy().tobytes()))
        else:
            blobs.append((name, list(t.shape), 0, t.numpy().astype("<f4").tobytes()))
    out = bytearray(b"GGUF") + struct.pack("<IQQ", 3, len(blobs), len(kvs))
    for key, typ, val in kvs:
        _gs(out, key)
        out += struct.pack("<I", GGUF_T[typ])
        if typ == "arr":
            et, items = val
            out += struct.pack("<IQ", GGUF_T[et], len(items))
            for it in items:
                _gv(out, et, it)
        else:
            _gv(out, typ, val)
    off = 0
    for name, shape, ty, data in blobs:
        _gs(out, name)
        ne = list(reversed(shape))
        out += struct.pack("<I", len(ne)) + b"".join(struct.pack("<Q", d) for d in ne)
        out += struct.pack("<IQ", ty, off)
        off += len(data) + (-len(data)) % align
    out += b"\0" * ((-len(out)) % align)
    for _, _, _, data in blobs:
        out += data + b"\0" * ((-len(data)) % align)
    with open(path, "wb") as f:
        f.write(out)


def gguf_metadata(name):
    pieces = [SP.id_to_piece(i) for i in range(V)]
    types = [2 if SP.is_unknown(i) else 3 if SP.is_control(i) else 6 if SP.is_byte(i) else 5 if SP.is_unused(i) else 1 for i in range(V)]
    return [
        ("general.architecture", "str", "llama"), ("general.name", "str", name), ("general.file_type", "u32", 1),
        ("llama.context_length", "u32", CTX), ("llama.embedding_length", "u32", D), ("llama.block_count", "u32", LAYERS),
        ("llama.feed_forward_length", "u32", FFN), ("llama.attention.head_count", "u32", HEADS),
        ("llama.attention.head_count_kv", "u32", HEADS), ("llama.rope.dimension_count", "u32", D // HEADS),
        ("llama.rope.freq_base", "f32", 10000.0), ("llama.attention.layer_norm_rms_epsilon", "f32", 1e-6),
        ("llama.vocab_size", "u32", V),
        ("tokenizer.ggml.model", "str", "llama"), ("tokenizer.ggml.tokens", "arr", ("str", pieces)),
        ("tokenizer.ggml.scores", "arr", ("f32", [SP.get_score(i) for i in range(V)])),
        ("tokenizer.ggml.token_type", "arr", ("i32", types)),
        ("tokenizer.ggml.unknown_token_id", "u32", 0), ("tokenizer.ggml.bos_token_id", "u32", 1),
        ("tokenizer.ggml.eos_token_id", "u32", EOS), ("tokenizer.ggml.padding_token_id", "u32", PAD),
        ("tokenizer.ggml.add_bos_token", "bool", False), ("tokenizer.ggml.add_eos_token", "bool", False),
        ("tokenizer.ggml.add_space_prefix", "bool", True),
    ]


def rope_permute(w):
    return w.reshape(HEADS, 2, w.shape[0] // HEADS // 2, w.shape[1]).transpose(1, 2).reshape(w.shape)


def gguf_tensors(model):
    sd = {k: v.detach().float().cpu() for k, v in model.state_dict().items()}
    T = {"token_embd.weight": sd["model.embed_tokens.weight"], "output_norm.weight": sd["model.norm.weight"]}
    for i in range(LAYERS):
        p, b = f"model.layers.{i}.", f"blk.{i}."
        T.update({b + "attn_norm.weight": sd[p + "input_layernorm.weight"],
                  b + "attn_q.weight": rope_permute(sd[p + "self_attn.q_proj.weight"]),
                  b + "attn_k.weight": rope_permute(sd[p + "self_attn.k_proj.weight"]),
                  b + "attn_v.weight": sd[p + "self_attn.v_proj.weight"],
                  b + "attn_output.weight": sd[p + "self_attn.o_proj.weight"],
                  b + "ffn_norm.weight": sd[p + "post_attention_layernorm.weight"],
                  b + "ffn_gate.weight": sd[p + "mlp.gate_proj.weight"],
                  b + "ffn_up.weight": sd[p + "mlp.up_proj.weight"],
                  b + "ffn_down.weight": sd[p + "mlp.down_proj.weight"]})
    return T


def llamacpp_style_logits(T, ids):
    hd, L = D // HEADS, len(ids)
    rms = lambda x, w: x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + 1e-6) * w
    ang = torch.arange(L)[:, None].float() * (1.0 / 10000 ** (torch.arange(0, hd, 2).float() / hd))[None]
    cos, sin = ang.cos()[:, None], ang.sin()[:, None]

    def rope(q):
        a, b = q.view(L, HEADS, hd // 2, 2).unbind(-1)
        return torch.stack([a * cos - b * sin, a * sin + b * cos], -1).reshape(L, HEADS, hd)

    x = T["token_embd.weight"][torch.tensor(ids)]
    mask = torch.triu(torch.full((L, L), float("-inf")), 1)
    for i in range(LAYERS):
        b = f"blk.{i}."
        h = rms(x, T[b + "attn_norm.weight"])
        q, k = rope((h @ T[b + "attn_q.weight"].T).view(L, HEADS, hd)), rope((h @ T[b + "attn_k.weight"].T).view(L, HEADS, hd))
        v = (h @ T[b + "attn_v.weight"].T).view(L, HEADS, hd)
        att = torch.softmax(torch.einsum("qhd,khd->hqk", q, k) / hd ** 0.5 + mask, -1)
        x = x + torch.einsum("hqk,khd->qhd", att, v).reshape(L, D) @ T[b + "attn_output.weight"].T
        h = rms(x, T[b + "ffn_norm.weight"])
        x = x + (F.silu(h @ T[b + "ffn_gate.weight"].T) * (h @ T[b + "ffn_up.weight"].T)) @ T[b + "ffn_down.weight"].T
    return rms(x, T["output_norm.weight"]) @ T["token_embd.weight"].T


EXPORT_README = '''# Small story models (about 17M parameters each)

- `simple-first.gguf`: toki pona first, then a bridge, then a small English budget
- `english.gguf`: the same small English budget only
- `ceiling.gguf`: lots of fresh English (an upper bound)

They write simple English stories (trained on SimpleStories). They are not chat models:
give them the start of a story and they continue it.

    ollama create simple-first -f Modelfile.simple-first
    ollama run simple-first "Once upon a time, a girl named Kim"

`tok.model` is the SentencePiece tokenizer, and `hf-*/` holds the same models in Hugging Face format.
'''


def modelfile(name):
    return f'''FROM ./{name}.gguf
TEMPLATE """{{{{ .Prompt }}}}"""
PARAMETER temperature 0.7
PARAMETER top_k 40
PARAMETER num_ctx {CTX}
PARAMETER num_predict 200
'''


def export(models, out="real_export"):
    os.makedirs(out, exist_ok=True)
    checks = {}
    ids = [EOS] + SP.encode("Once upon a time, there was a little cat who wanted to fly.")
    for name, model in models.items():
        model.eval()
        T = gguf_tensors(model)
        with torch.no_grad():
            ref = model.float()(input_ids=torch.tensor([ids], device=DEV)).logits[0].float().cpu()
        checks[name] = (ref - llamacpp_style_logits(T, ids)).abs().max().item()
        write_gguf(f"{out}/{name}.gguf", gguf_metadata(name), list(T.items()))
        open(f"{out}/Modelfile.{name}", "w").write(modelfile(name))
        model.save_pretrained(f"{out}/hf-{name}")
    os.system(f"cp tok.model {out}/")
    open(f"{out}/README.md", "w").write(EXPORT_README)
    with zipfile.ZipFile(f"{out}.zip", "w", zipfile.ZIP_DEFLATED) as z:
        for root, _, files in os.walk(out):
            for fn in files:
                z.write(os.path.join(root, fn))
    log(f"export check (max logit diff, HF vs llama.cpp-style): {checks}; wrote {out}.zip ({os.path.getsize(out + '.zip')/1e6:.0f} MB)")
    return checks

## Run everything

In [ ]:
def main():
    log(f"device {DEV}, vocab {V}, params {sum(p.numel() for p in make_model().parameters())/1e6:.1f}M")
    R, models = {}, {}
    for cond in ("simple-first", "english", "ceiling"):
        log(f"=== {cond} ===")
        model, curve = train_condition(cond)
        res = {"val_loss": val_loss(model), "curve": curve}
        if cond == "simple-first":
            res["tp_val_loss"] = val_loss(model, TPVAL_WIN)
        log("  BLiMP"); res["blimp"] = blimp(model)
        log("  bAbI"); res["babi"] = babi(model)
        res["samples"] = samples(model)
        log(f"  {cond}: val {res['val_loss']:.3f}  BLiMP {res['blimp']['overall']*100:.1f}%  bAbI {res['babi']['mean']*100:.1f}%")
        for s in res["samples"]:
            print("   >", s[:300].replace("\n", " "))
        R[cond] = res
        models[cond] = model.cpu()
        json.dump(R, open("real_results.json", "w"), indent=1)
        torch.cuda.empty_cache() if DEV == "cuda" else None
    for m in models.values():
        m.to(DEV)
    checks = export(models)
    report(R, checks)


def report(R, checks):
    pct = lambda v: f"{v*100:.1f}%"
    conds = list(R)
    L = ["# Simple language first, real data: results", "",
         f"One seed. English budget: {N_STORIES:,} SimpleStories + {len(pairs):,} Tatoeba English sentences "
         f"({len(S['en'])/1e6:.1f}M tokens). Ceiling: {len(S['ceil'])/1e6:.0f}M tokens of fresh English. "
         f"{S1 + S2 + S3:,} training steps each, {BATCH}×{CTX} tokens per step.", "",
         "| model | English val loss (lower is better) | BLiMP | bAbI (mean of 20 tasks) |", "|---|---|---|---|"]
    for c in conds:
        L.append(f"| {c} | {R[c]['val_loss']:.3f} | {pct(R[c]['blimp']['overall'])} | {pct(R[c]['babi']['mean'])} |")
    fields = [k for k in R[conds[0]]["blimp"] if k != "overall"]
    L += ["", "## BLiMP by area", "", "| model | " + " | ".join(fields) + " |", "|---" * (len(fields) + 1) + "|"]
    for c in conds:
        L.append(f"| {c} | " + " | ".join(pct(R[c]["blimp"][f]) for f in fields) + " |")
    tasks = [k for k in R[conds[0]]["babi"] if k != "mean"]
    L += ["", "## bAbI by task (chance is roughly 1 in the number of possible answers)", "",
          "| model | " + " | ".join(t.replace("task", "") for t in tasks) + " |", "|---" * (len(tasks) + 1) + "|"]
    for c in conds:
        L.append(f"| {c} | " + " | ".join(f"{R[c]['babi'][t]*100:.0f}" for t in tasks) + " |")
    if "tp_val_loss" in R.get("simple-first", {}):
        L += ["", f"simple-first's toki pona val loss at the end: {R['simple-first']['tp_val_loss']:.3f}"]
    L += ["", "## English val loss over training", ""]
    for c in conds:
        L.append(f"- {c}: " + ", ".join(f"{s}:{v:.2f}" for s, v in R[c]["curve"]))
    L += ["", "## Samples (prompt: Once upon a time / The little dog / Mia wanted to)", ""]
    for c in conds:
        L.append(f"**{c}**")
        L += [f"> {s.replace(chr(10), ' ')}" for s in R[c]["samples"]] + [""]
    L += [f"Export check (max logit difference HF vs GGUF-style forward): " + ", ".join(f"{k} {v:.1e}" for k, v in checks.items())]
    text = "\n".join(L) + "\n"
    open("REAL_RESULTS.md", "w").write(text)
    print(text)

In [ ]:
if __name__ == "__main__" or "ipykernel" in sys.modules:
    main()